# TCGA-PRAD: reproducible expression and clinical data

This report reads the **`tcga-prad-v2`** outputs and the raw-source manifest. It makes no network requests. Counts are calculated from these files.

Before running this notebook, run `uv run --locked tcga-prad` from the repository root. The output layout matches the Taylor report (`taylor_gse21034_report.ipynb`); see `docs/tcga_prad_data_dictionary.md` for every column.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
SNAPSHOT = 'tcga-prad-v2'
RAW = ROOT / 'data/raw' / SNAPSHOT
OUT = ROOT / 'data/processed' / SNAPSHOT
ANNDATA = ROOT / 'data/anndata' / SNAPSHOT
ids = {c: 'string' for c in ['sample_id', 'cbio_sample_id', 'patient_id', 'sample_type_code', 'feature_id', 'gene_id']}
manifest = json.loads((RAW / 'manifest.json').read_text())
metadata = json.loads((OUT / 'metadata.json').read_text())
validation = json.loads((OUT / 'validation.json').read_text())
expression = pd.read_csv(OUT / 'expression.csv.gz', index_col=0, dtype={'feature_id': 'string'}, float_precision='round_trip')
clinical = pd.read_csv(OUT / 'clinical.csv', dtype=ids, low_memory=False)
samples = pd.read_csv(OUT / 'samples.csv', dtype='string')
genes = pd.read_csv(OUT / 'genes.csv', dtype=ids)
feature_audit = pd.read_csv(OUT / 'feature_audit.csv', dtype=ids)
audit = pd.read_csv(OUT / 'sample_audit.csv', dtype=ids)
print('Expression (features x samples):', expression.shape)

## 1. Sources and integrity

The RSEM matrix, the clinical attribute definitions, the paged clinical records, the sample inventory and the legacy Gleason records are all cached. The manifest gives each one's URL, retrieval time and SHA-256. This cell re-verifies every raw file and every final file.

In [ ]:
sources = pd.DataFrame(manifest['sources'].values())
display(sources[['path', 'url', 'retrieved_at_utc', 'bytes']])
def digest(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()
assert all(digest(RAW / e['path']) == e['sha256'] for e in manifest['sources'].values())
assert digest(RAW / 'manifest.json') == metadata['source_manifest_sha256']
for folder in (OUT, ANNDATA):
    for line in (folder / 'checksums.sha256').read_text().splitlines():
        expected, filename = line.split('  ', 1)
        assert digest(folder / filename) == expected, filename
print('All raw and final file checksums verified.')

## 2. Sample selection and patient matching

Barcode characters 1–12 identify the patient and characters 14–15 the sample type. The pipeline keeps type `01` (primary tumor) and one sample per patient, then removes expression QC outliers. Each barcode's patient is checked against the cBioPortal sample inventory, and cBioPortal `SAMPLE_TYPE` must be `Primary`.

In [ ]:
display(audit.groupby(['selected', 'selection_reason']).size().rename('samples').to_frame())
display(audit.loc[~audit.selected, ['sample_id', 'selection_reason', 'median_log2', 'genes_detected']])
kept = audit[audit.selected]
display(kept[['median_log2', 'genes_detected']].describe().T)
print('Distinct selected samples:', samples.sample_id.nunique(), '| distinct patients:', samples.patient_id.nunique())

## 3. Expression processing

The source values are PanCanAtlas batch-corrected RSEM counts. The pipeline clips negative values to 0, applies log2(x + 1) and rounds to 4 decimals. Rows with a missing value in any selected sample are removed, and no rows are collapsed. The unclipped source values are in the h5ad `rsem` layer.

In [ ]:
display(pd.Series(metadata['expression'], name='policy').to_frame())
display(feature_audit.reason.value_counts().rename('source rows').to_frame())
display(pd.DataFrame({'log2(RSEM + 1)': np.quantile(expression.values, [0, .25, .5, .75, 1])},
                     index=['minimum', '25%', 'median', '75%', 'maximum']))

### Rows sharing an Entrez ID stay separate

When an Entrez ID repeats in the source, each row's `feature_id` gets the source row number appended. The cell below recomputes one repeated pair from the raw RSEM file and confirms that the two rows hold different measurements.

In [ ]:
raw = pd.read_csv(RAW / 'data_mrna_seq_v2_rsem.txt', sep='\t', low_memory=False)
pair = genes[genes.feature_id.str.contains('_row')].groupby('gene_id').filter(lambda g: len(g) == 2).head(2)
display(pair)
for _, row in pair.iterrows():
    source = raw.iloc[int(row.source_row) - 1][expression.columns].astype(float)
    np.testing.assert_array_equal(np.log2(source.clip(lower=0) + 1).round(4).to_numpy(),
                                  expression.loc[row.feature_id].to_numpy())
print('Correlation between the two rows:', round(np.corrcoef(expression.loc[pair.feature_id].values)[0, 1], 3))

## 4. Endpoints

**PFI** is the primary endpoint. OS, DSS and DFI are secondary. Each endpoint has `<e>_event`, `<e>_months`, `<e>_eligible` and `<e>_eligibility_reason` columns, decoded like Taylor `dfs_*`: missing values stay missing, and they are never treated as censored.

In [ ]:
display(pd.DataFrame(validation['endpoints']).T)
display(pd.DataFrame({e: v['status_codes'] for e, v in metadata['endpoints'].items()}).T)
display(clinical[['sample_id', 'patient__PFS_STATUS', 'patient__PFS_MONTHS', 'pfi_event', 'pfi_months', 'pfi_eligible']].head(8))
print('Median observed time mixes event and censoring times; it is not a survival median.')

## 5. Clinical covariates

Source fields keep their `patient__`, `sample__` and `legacy_patient__` prefixes. Three derived covariates are added: `age`, `path_t_group` and `gleason_score`.

In [ ]:
display(clinical[['age', 'gleason_score']].describe().T)
display(clinical.path_t_group.value_counts(dropna=False).rename('samples').to_frame())
display(pd.crosstab(clinical.gleason_score, clinical.pfi_event, margins=True))
missing = clinical.isna().sum()
display(missing[missing > 0].rename('missing values').to_frame())

## 6. Alignment and AnnData

The cell below checks the same invariants as the pipeline's validation, independently of the saved report.

In [ ]:
checks = {
    'sample_order': expression.columns.tolist() == samples.sample_id.tolist() == clinical.sample_id.tolist(),
    'feature_order': expression.index.tolist() == genes.feature_id.tolist(),
    'unique_sample_ids': samples.sample_id.is_unique,
    'unique_patient_ids': samples.patient_id.is_unique,
    'unique_feature_ids': expression.index.is_unique,
    'finite_expression': bool(np.isfinite(expression.values).all()),
    'every_source_row_audited': len(feature_audit) == validation['source_features'],
    'selected_samples_audited': set(audit.loc[audit.selected, 'sample_id']) == set(samples.sample_id),
    'eligible_pfi_complete': bool(clinical.loc[clinical.pfi_eligible, ['pfi_event', 'pfi_months']].notna().all().all()),
}
display(pd.Series(checks, name='passed').to_frame())
assert all(checks.values())
import anndata as ad
adata = ad.read_h5ad(ANNDATA / 'expression.h5ad')
assert adata.obs_names.tolist() == expression.columns.tolist()
assert adata.var_names.tolist() == expression.index.tolist()
np.testing.assert_array_equal(adata.X, expression.to_numpy().T)
display(adata)

## Remaining interpretation limits

The structural checks above don't establish that the data suit a particular analysis. In particular:

- Clipping and the log transform are modeling choices.
- PFI is not BCR.
- Gleason comes from a second cBioPortal study.

In [ ]:
display(pd.Series(metadata['limitations'], name='documented_limitations').to_frame())
display(pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size} for p in sorted(OUT.iterdir())]))